In [1]:
import pandas as pd
from typing import Any

dataset = pd.read_json("dataset.json")
days: list[Any] = list(dataset['fecha'])
hours: list[Any] = list(dataset.iloc[0]['horas'].keys())
hours.sort(key=lambda x: int(x.split(':')[0]))
variables: list[Any] = list(dataset.iloc[0]['horas'][hours[0]])

df = pd.DataFrame(columns=['Día', 'Hora'] + variables)

In [2]:
rows = []
for day_index, date in enumerate(days):
    for hour in hours:
        row = [date, hour]
        for var in variables:
            value = dataset.iloc[day_index]['horas'][hour][var]
            row.append(float(value) if value != '' else None)
        rows.append(row)

df = pd.DataFrame(rows, columns=['Día', 'Hora'] + variables)

filas_completas = df.dropna().shape[0]
print(f"Filas sin None: {filas_completas} de {len(df)} ({(filas_completas/len(df))*100:.1f}%)")

print("\nDatos completos por variable:")
print(df[variables].notna().sum())

Filas sin None: 10 de 8784 (0.1%)

Datos completos por variable:
CO      8426
NO2     6875
NO      6869
NxOy    6875
O3      8116
P2.5    8423
VV      8478
HR      8500
T        654
PB      8123
RS      8500
PP      7860
dtype: int64


In [3]:
# Eliminar columna T
df = df.drop(columns=['T'])

# Reemplazar valores negativos por 0 en variables que no pueden ser negativas
variables_no_negativas = ['CO', 'NO2', 'NO', 'NxOy', 'O3', 'P2.5', 'PP']
for var in variables_no_negativas:
    if var in df.columns:
        df[var] = df[var].apply(lambda x: 0 if pd.notna(x) and x < 0 else x)

# Reemplazar outliers extremos en VV (velocidad del viento > 500 es imposible)
df.loc[df['VV'] > 500, 'VV'] = None

print("✓ Columna T eliminada")
print("✓ Valores negativos reemplazados por 0")
print("✓ Outliers en VV corregidos")

print(f"\nFilas completas después de limpieza: {df.dropna().shape[0]} de {len(df)} ({(df.dropna().shape[0]/len(df))*100:.1f}%)")

print("\nDatos completos por variable:")
for var in df.columns[2:]:  # Excluir 'Día' y 'Hora'
    count = df[var].notna().sum()
    total = len(df)
    percentage = (count/total)*100
    print(f"{var:<10} {count:>5}/{total} ({percentage:>5.1f}%)")

✓ Columna T eliminada
✓ Valores negativos reemplazados por 0
✓ Outliers en VV corregidos

Filas completas después de limpieza: 5813 de 8784 (66.2%)

Datos completos por variable:
CO          8426/8784 ( 95.9%)
NO2         6875/8784 ( 78.3%)
NO          6869/8784 ( 78.2%)
NxOy        6875/8784 ( 78.3%)
O3          8116/8784 ( 92.4%)
P2.5        8423/8784 ( 95.9%)
VV          8477/8784 ( 96.5%)
HR          8500/8784 ( 96.8%)
PB          8123/8784 ( 92.5%)
RS          8500/8784 ( 96.8%)
PP          7860/8784 ( 89.5%)


In [4]:
import numpy as np

# Extraer features temporales
df['Día_numérico'] = pd.to_datetime(df['Día']).dt.day
df['Mes'] = pd.to_datetime(df['Día']).dt.month
df['Hora_numérica'] = df['Hora'].str.split(':').str[0].astype(int)

print("✓ Features temporales creadas")
print(f"  - Día numérico: 1-31")
print(f"  - Mes: {df['Mes'].min()}-{df['Mes'].max()}")
print(f"  - Hora numérica: 0-23")

✓ Features temporales creadas
  - Día numérico: 1-31
  - Mes: 1-12
  - Hora numérica: 0-23


In [5]:
# Variables a usar (excluyendo T que ya fue eliminada)
var_cols = ['CO', 'NO2', 'NO', 'NxOy', 'O3', 'P2.5', 'VV', 'HR', 'PB', 'RS', 'PP']

# Crear dataset con rezagos
def create_lagged_features(df, lag_hours=5):
    df_lagged = df.copy()
    
    # Crear rezagos para cada variable
    for var in var_cols:
        for lag in range(1, lag_hours + 1):
            df_lagged[f'{var}_lag{lag}'] = df[var].shift(lag)
    
    # Crear target (próxima hora para cada variable)
    for var in var_cols:
        df_lagged[f'{var}_next'] = df[var].shift(-1)
    
    # Eliminar filas sin rezagos completos o sin target
    df_lagged = df_lagged.dropna()
    
    return df_lagged

print("Creando features con rezagos de 5 horas...")
df_lagged = create_lagged_features(df, lag_hours=5)

print(f"✓ Dataset con rezagos creado")
print(f"  Filas originales: {len(df)}")
print(f"  Filas con rezagos completos: {len(df_lagged)}")
print(f"  Filas perdidas: {len(df) - len(df_lagged)}")

Creando features con rezagos de 5 horas...
✓ Dataset con rezagos creado
  Filas originales: 8784
  Filas con rezagos completos: 5645
  Filas perdidas: 3139


In [6]:
# Split temporal: 70% train, 15% validation, 15% test
n_total = len(df_lagged)
train_size = int(0.70 * n_total)
val_size = int(0.15 * n_total)

train_df = df_lagged.iloc[:train_size]
val_df = df_lagged.iloc[train_size:train_size + val_size]
test_df = df_lagged.iloc[train_size + val_size:]

print("Split temporal completado:")
print(f"  Train: {len(train_df)} filas ({len(train_df)/len(df_lagged)*100:.1f}%)")
print(f"  Val:   {len(val_df)} filas ({len(val_df)/len(df_lagged)*100:.1f}%)")
print(f"  Test:  {len(test_df)} filas ({len(test_df)/len(df_lagged)*100:.1f}%)")
print(f"\nPeriodos aproximados:")
print(f"  Train: {train_df['Día'].iloc[0]} a {train_df['Día'].iloc[-1]}")
print(f"  Val:   {val_df['Día'].iloc[0]} a {val_df['Día'].iloc[-1]}")
print(f"  Test:  {test_df['Día'].iloc[0]} a {test_df['Día'].iloc[-1]}")

Split temporal completado:
  Train: 3951 filas (70.0%)
  Val:   846 filas (15.0%)
  Test:  848 filas (15.0%)

Periodos aproximados:
  Train: 2024-05-28 a 2024-12-13
  Val:   2024-12-13 a 2025-01-17
  Test:  2025-01-17 a 2025-02-25


In [7]:
# Separar features (X) y targets (y)

# Features: 
# - Variables actuales (11)
# - Rezagos de 5 horas para 11 variables (55)
# - Features temporales: Hora, Día, Mes (3)
# Total: 69 features

feature_cols = (
    var_cols +  # Variables actuales
    [f'{var}_lag{i}' for var in var_cols for i in range(1, 6)] +  # Rezagos
    ['Hora_numérica', 'Día_numérico', 'Mes']  # Temporales
)

target_cols = [f'{var}_next' for var in var_cols]  # 11 targets (próxima hora)

# Train
X_train = train_df[feature_cols]
y_train = train_df[target_cols]

# Validation
X_val = val_df[feature_cols]
y_val = val_df[target_cols]

# Test
X_test = test_df[feature_cols]
y_test = test_df[target_cols]

print("Conjuntos de datos creados:")
print(f"\n{'Set':<10} {'X shape':<20} {'y shape':<20}")
print("-" * 50)
print(f"{'Train':<10} {str(X_train.shape):<20} {str(y_train.shape):<20}")
print(f"{'Val':<10} {str(X_val.shape):<20} {str(y_val.shape):<20}")
print(f"{'Test':<10} {str(X_test.shape):<20} {str(y_test.shape):<20}")
print(f"\n✓ Total features: {len(feature_cols)}")
print(f"✓ Total targets: {len(target_cols)}")

Conjuntos de datos creados:

Set        X shape              y shape             
--------------------------------------------------
Train      (3951, 69)           (3951, 11)          
Val        (846, 69)            (846, 11)           
Test       (848, 69)            (848, 11)           

✓ Total features: 69
✓ Total targets: 11


In [8]:
# Analizar cobertura de meses y distribución de datos

print("="*70)
print("ANÁLISIS DE COBERTURA TEMPORAL")
print("="*70)

# Datos con rezagos completos
print(f"\n📊 Dataset con rezagos (df_lagged):")
print(f"   Total filas: {len(df_lagged)}")
print(f"   Periodo: {df_lagged['Día'].iloc[0]} a {df_lagged['Día'].iloc[-1]}")

# Análisis por mes
df_lagged['Fecha'] = pd.to_datetime(df_lagged['Día'])
monthly_counts = df_lagged.groupby(df_lagged['Fecha'].dt.to_period('M')).size()

print(f"\n📅 Distribución por mes:")
print(f"{'Mes':<15} {'Filas':<10} {'Días aprox':<15} {'% del total':<15}")
print("-" * 70)
for period, count in monthly_counts.items():
    percentage = (count / len(df_lagged)) * 100
    days_approx = count / 24  # Asumiendo 24 horas por día
    print(f"{str(period):<15} {count:<10} {days_approx:<15.1f} {percentage:<15.1f}%")

print(f"\n📈 Estadísticas:")
print(f"   Meses representados: {len(monthly_counts)}")
print(f"   Meses completos (>600 filas/25 días): {sum(monthly_counts > 600)}")
print(f"   Meses parciales (<600 filas): {sum(monthly_counts <= 600)}")

ANÁLISIS DE COBERTURA TEMPORAL

📊 Dataset con rezagos (df_lagged):
   Total filas: 5645
   Periodo: 2024-05-28 a 2025-02-25

📅 Distribución por mes:
Mes             Filas      Días aprox      % del total    
----------------------------------------------------------------------
2024-05         4          0.2             0.1            %
2024-06         381        15.9            6.7            %
2024-07         576        24.0            10.2           %
2024-08         680        28.3            12.0           %
2024-09         706        29.4            12.5           %
2024-10         596        24.8            10.6           %
2024-11         706        29.4            12.5           %
2024-12         744        31.0            13.2           %
2025-01         744        31.0            13.2           %
2025-02         508        21.2            9.0            %

📈 Estadísticas:
   Meses representados: 10
   Meses completos (>600 filas/25 días): 5
   Meses parciales (<600 filas): 5

In [9]:
# Analizar splits actuales
print("\n" + "="*70)
print("ANÁLISIS DE SPLITS ACTUALES (70-15-15)")
print("="*70)

def analyze_split(df_split, name):
    months = df_split.groupby(pd.to_datetime(df_split['Día']).dt.to_period('M')).size()
    print(f"\n{name}:")
    print(f"   Filas: {len(df_split)}")
    print(f"   Periodo: {df_split['Día'].iloc[0]} a {df_split['Día'].iloc[-1]}")
    print(f"   Meses cubiertos: {list(months.index.astype(str))}")
    return months

train_months = analyze_split(train_df, "TRAIN")
val_months = analyze_split(val_df, "VALIDATION")
test_months = analyze_split(test_df, "TEST")

# Identificar problema
print("\n" + "="*70)
print("⚠️  PROBLEMAS IDENTIFICADOS:")
print("="*70)
print("\n1. COBERTURA ESTACIONAL INCOMPLETA:")
print("   - Train NO incluye: Enero, Febrero (invierno/verano según hemisferio)")
print("   - El modelo nunca vio patrones de esos meses durante entrenamiento")
print("\n2. DATOS FRAGMENTADOS:")
print(f"   - Mayo 2024: Solo {monthly_counts['2024-05']} filas (casi vacío)")
print(f"   - Junio 2024: Solo {monthly_counts['2024-06']} filas (parcial)")
print(f"   - Febrero 2025: Solo {monthly_counts['2025-02']} filas (parcial)")
print("\n3. SIN GAP ENTRE SPLITS:")
print("   - Con rezagos de 5 horas, podría haber mínima fuga de información")


ANÁLISIS DE SPLITS ACTUALES (70-15-15)

TRAIN:
   Filas: 3951
   Periodo: 2024-05-28 a 2024-12-13
   Meses cubiertos: ['2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']

VALIDATION:
   Filas: 846
   Periodo: 2024-12-13 a 2025-01-17
   Meses cubiertos: ['2024-12', '2025-01']

TEST:
   Filas: 848
   Periodo: 2025-01-17 a 2025-02-25
   Meses cubiertos: ['2025-01', '2025-02']

⚠️  PROBLEMAS IDENTIFICADOS:

1. COBERTURA ESTACIONAL INCOMPLETA:
   - Train NO incluye: Enero, Febrero (invierno/verano según hemisferio)
   - El modelo nunca vio patrones de esos meses durante entrenamiento

2. DATOS FRAGMENTADOS:
   - Mayo 2024: Solo 4 filas (casi vacío)
   - Junio 2024: Solo 381 filas (parcial)
   - Febrero 2025: Solo 508 filas (parcial)

3. SIN GAP ENTRE SPLITS:
   - Con rezagos de 5 horas, podría haber mínima fuga de información


In [10]:
# ============================================================================
# ESTRATEGIA MEJORADA: SPLIT CON GAP Y MEJOR COBERTURA ESTACIONAL
# ============================================================================

print("\n" + "="*70)
print("NUEVA ESTRATEGIA: ROLLING ORIGIN CON GAP")
print("="*70)

# Parámetros
LAG_HOURS = 5
GAP_HOURS = 24  # 1 día completo de gap para evitar cualquier fuga

# Filtrar datos más completos (desde Junio 2024)
df_filtered = df_lagged[pd.to_datetime(df_lagged['Día']) >= '2024-06-01'].reset_index(drop=True)

print(f"\n✓ Dataset filtrado (desde Junio 2024):")
print(f"   Filas: {len(df_filtered)}")
print(f"   Periodo: {df_filtered['Día'].iloc[0]} a {df_filtered['Día'].iloc[-1]}")

# Calcular índices con GAP
n_total = len(df_filtered)
train_size = int(0.70 * n_total)
val_size = int(0.15 * n_total)

# Índices de corte
train_end_idx = train_size
gap1_end_idx = train_end_idx + GAP_HOURS
val_end_idx = gap1_end_idx + val_size
gap2_end_idx = val_end_idx + GAP_HOURS
test_start_idx = gap2_end_idx

# Crear splits con GAP
train_df_gap = df_filtered.iloc[:train_end_idx]
val_df_gap = df_filtered.iloc[gap1_end_idx:val_end_idx]
test_df_gap = df_filtered.iloc[test_start_idx:]

print(f"\n📊 SPLITS CON GAP DE {GAP_HOURS} HORAS:")
print(f"{'Split':<15} {'Filas':<10} {'Inicio':<15} {'Fin':<15} {'Meses'}")
print("-" * 90)

def get_months(df_split):
    months = df_split.groupby(pd.to_datetime(df_split['Día']).dt.to_period('M')).size()
    return ', '.join([str(m) for m in months.index.astype(str)])

print(f"{'Train':<15} {len(train_df_gap):<10} {train_df_gap['Día'].iloc[0]:<15} {train_df_gap['Día'].iloc[-1]:<15} {get_months(train_df_gap)}")
print(f"{'[GAP]':<15} {GAP_HOURS:<10} {'-':<15} {'-':<15} (protección)")
print(f"{'Validation':<15} {len(val_df_gap):<10} {val_df_gap['Día'].iloc[0]:<15} {val_df_gap['Día'].iloc[-1]:<15} {get_months(val_df_gap)}")
print(f"{'[GAP]':<15} {GAP_HOURS:<10} {'-':<15} {'-':<15} (protección)")
print(f"{'Test':<15} {len(test_df_gap):<10} {test_df_gap['Día'].iloc[0]:<15} {test_df_gap['Día'].iloc[-1]:<15} {get_months(test_df_gap)}")

print(f"\n✓ Sin data leakage: GAP de {GAP_HOURS} horas entre cada split")
print(f"✓ Orden cronológico preservado")
print(f"✓ Cobertura: Junio 2024 - Febrero 2025 (9 meses)")


NUEVA ESTRATEGIA: ROLLING ORIGIN CON GAP

✓ Dataset filtrado (desde Junio 2024):
   Filas: 5641
   Periodo: 2024-06-13 a 2025-02-25

📊 SPLITS CON GAP DE 24 HORAS:
Split           Filas      Inicio          Fin             Meses
------------------------------------------------------------------------------------------
Train           3948       2024-06-13      2024-12-13      2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12
[GAP]           24         -               -               (protección)
Validation      846        2024-12-14      2025-01-18      2024-12, 2025-01
[GAP]           24         -               -               (protección)
Test            799        2025-01-19      2025-02-25      2025-01, 2025-02

✓ Sin data leakage: GAP de 24 horas entre cada split
✓ Orden cronológico preservado
✓ Cobertura: Junio 2024 - Febrero 2025 (9 meses)


In [25]:
# Actualizar X, y con los nuevos splits
print("\n" + "="*70)
print("ACTUALIZANDO CONJUNTOS DE DATOS")
print("="*70)

# Train
X_train = train_df_gap[feature_cols]
y_train = train_df_gap[target_cols]

# Validation
X_val = val_df_gap[feature_cols]
y_val = val_df_gap[target_cols]

# Test
X_test = test_df_gap[feature_cols]
y_test = test_df_gap[target_cols]

print(f"\n{'Set':<15} {'X shape':<20} {'y shape':<20}")
print("-" * 55)
print(f"{'Train':<15} {str(X_train.shape):<20} {str(y_train.shape):<20}")
print(f"{'Validation':<15} {str(X_val.shape):<20} {str(y_val.shape):<20}")
print(f"{'Test':<15} {str(X_test.shape):<20} {str(y_test.shape):<20}")

print(f"\n✅ Conjuntos actualizados con GAP y sin data leakage")
print(f"✅ Features: {len(feature_cols)} (11 vars + 55 lags + 3 temporales)")
print(f"✅ Targets: {len(target_cols)} (predicción de 11 variables)")

# Advertencia sobre cobertura estacional
print(f"\n⚠️  NOTA IMPORTANTE SOBRE COBERTURA ESTACIONAL:")
print(f"   - Los datos disponibles cubren Jun-Feb (9 meses)")
print(f"   - Faltan: Mar, Abr, May completos")
print(f"   - El modelo tendrá MENOR precisión en esos meses no vistos")
print(f"   - Recomendación: Reentrenar cuando tengas datos de todo el año")


ACTUALIZANDO CONJUNTOS DE DATOS

Set             X shape              y shape             
-------------------------------------------------------
Train           (3948, 69)           (3948, 11)          
Validation      (846, 69)            (846, 11)           
Test            (799, 69)            (799, 11)           

✅ Conjuntos actualizados con GAP y sin data leakage
✅ Features: 69 (11 vars + 55 lags + 3 temporales)
✅ Targets: 11 (predicción de 11 variables)

⚠️  NOTA IMPORTANTE SOBRE COBERTURA ESTACIONAL:
   - Los datos disponibles cubren Jun-Feb (9 meses)
   - Faltan: Mar, Abr, May completos
   - El modelo tendrá MENOR precisión en esos meses no vistos
   - Recomendación: Reentrenar cuando tengas datos de todo el año


In [11]:
# ============================================================================
# TIME SERIES CROSS-VALIDATION CON ROLLING ORIGIN
# ============================================================================

print("\n" + "="*70)
print("TIME SERIES CROSS-VALIDATION (ROLLING ORIGIN)")
print("="*70)

# Configuración
N_FOLDS = 5  # Número de folds de validación
GAP_HOURS = 24  # Gap entre train y validation
MIN_TRAIN_SIZE = 2000  # Mínimo de datos para entrenar

# Usar dataset filtrado
df_cv = df_filtered.copy()

def create_time_series_folds(df, n_folds=5, gap=24, min_train_size=2000):
    """
    Crea folds de validación con rolling origin.
    Cada fold tiene más datos de entrenamiento que el anterior.
    """
    folds = []
    n_total = len(df)
    
    # Calcular tamaño de cada fold de validación
    val_size = (n_total - min_train_size - (n_folds * gap)) // n_folds
    
    for fold_idx in range(n_folds):
        # Train: desde el inicio hasta el punto de corte
        train_end = min_train_size + (fold_idx * val_size)
        
        # Gap: zona de protección
        gap_end = train_end + gap
        
        # Validation: después del gap
        val_end = gap_end + val_size
        
        # Asegurar que no nos pasemos del dataset
        if val_end > n_total:
            break
            
        train_df = df.iloc[:train_end]
        val_df = df.iloc[gap_end:val_end]
        
        folds.append({
            'fold': fold_idx + 1,
            'train': train_df,
            'val': val_df,
            'train_period': (train_df['Día'].iloc[0], train_df['Día'].iloc[-1]),
            'val_period': (val_df['Día'].iloc[0], val_df['Día'].iloc[-1])
        })
    
    return folds

# Crear folds
cv_folds = create_time_series_folds(df_cv, n_folds=N_FOLDS, gap=GAP_HOURS, min_train_size=MIN_TRAIN_SIZE)

print(f"\n✓ Folds creados: {len(cv_folds)}")
print(f"✓ Gap entre train-val: {GAP_HOURS} horas")
print(f"\n{'Fold':<8} {'Train':<25} {'Val':<25} {'Train Size':<12} {'Val Size':<10}")
print("-" * 90)

for fold_info in cv_folds:
    fold_num = fold_info['fold']
    train_size = len(fold_info['train'])
    val_size = len(fold_info['val'])
    train_period = f"{fold_info['train_period'][0]} - {fold_info['train_period'][1]}"
    val_period = f"{fold_info['val_period'][0]} - {fold_info['val_period'][1]}"
    
    print(f"Fold {fold_num:<3} {train_period:<25} {val_period:<25} {train_size:<12} {val_size:<10}")


TIME SERIES CROSS-VALIDATION (ROLLING ORIGIN)

✓ Folds creados: 5
✓ Gap entre train-val: 24 horas

Fold     Train                     Val                       Train Size   Val Size  
------------------------------------------------------------------------------------------
Fold 1   2024-06-13 - 2024-09-16   2024-09-17 - 2024-10-23   2000         704       
Fold 2   2024-06-13 - 2024-10-22   2024-10-23 - 2024-11-22   2704         704       
Fold 3   2024-06-13 - 2024-11-20   2024-11-22 - 2024-12-21   3408         704       
Fold 4   2024-06-13 - 2024-12-20   2024-12-21 - 2025-01-19   4112         704       
Fold 5   2024-06-13 - 2025-01-18   2025-01-19 - 2025-02-21   4816         704       


In [27]:
# Visualizar cobertura temporal de cada fold
print("\n" + "="*70)
print("COBERTURA TEMPORAL POR FOLD")
print("="*70)

for fold_info in cv_folds:
    fold_num = fold_info['fold']
    train_df_fold = fold_info['train']
    val_df_fold = fold_info['val']
    
    # Obtener meses cubiertos
    train_months = train_df_fold.groupby(pd.to_datetime(train_df_fold['Día']).dt.to_period('M')).size()
    val_months = val_df_fold.groupby(pd.to_datetime(val_df_fold['Día']).dt.to_period('M')).size()
    
    train_months_str = ', '.join([str(m) for m in train_months.index.astype(str)])
    val_months_str = ', '.join([str(m) for m in val_months.index.astype(str)])
    
    print(f"\nFold {fold_num}:")
    print(f"  Train meses: {train_months_str}")
    print(f"  Val meses:   {val_months_str}")

print("\n✅ Cada fold tiene diferente cobertura temporal")
print("✅ Los folds posteriores tienen más datos de entrenamiento")
print("✅ Esto simula cómo el modelo mejora con más historia")


COBERTURA TEMPORAL POR FOLD

Fold 1:
  Train meses: 2024-06, 2024-07, 2024-08, 2024-09
  Val meses:   2024-09, 2024-10

Fold 2:
  Train meses: 2024-06, 2024-07, 2024-08, 2024-09, 2024-10
  Val meses:   2024-10, 2024-11

Fold 3:
  Train meses: 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11
  Val meses:   2024-11, 2024-12

Fold 4:
  Train meses: 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12
  Val meses:   2024-12, 2025-01

Fold 5:
  Train meses: 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12, 2025-01
  Val meses:   2025-01, 2025-02

✅ Cada fold tiene diferente cobertura temporal
✅ Los folds posteriores tienen más datos de entrenamiento
✅ Esto simula cómo el modelo mejora con más historia


In [12]:
# ============================================================================
# CONJUNTO DE TEST FINAL (NO TOCAR HASTA EL FINAL)
# ============================================================================

print("\n" + "="*70)
print("CONJUNTO DE TEST FINAL")
print("="*70)

# El test set es el último 15% de los datos, después de un gap
test_start_idx = len(df_cv) - int(0.15 * len(df_cv))
test_gap_start = test_start_idx - GAP_HOURS

# Train+Val para el modelo final (todo antes del test gap)
train_val_final = df_cv.iloc[:test_gap_start]
test_final = df_cv.iloc[test_start_idx:]

print(f"\nConjunto de TEST (reservado para evaluación final):")
print(f"  Tamaño: {len(test_final)} filas")
print(f"  Periodo: {test_final['Día'].iloc[0]} a {test_final['Día'].iloc[-1]}")
test_months_final = test_final.groupby(pd.to_datetime(test_final['Día']).dt.to_period('M')).size()
test_months_str = ', '.join([str(m) for m in test_months_final.index.astype(str)])
print(f"  Meses: {test_months_str}")

print(f"\nDatos disponibles para Train+Val (CV):")
print(f"  Tamaño: {len(train_val_final)} filas")
print(f"  Periodo: {train_val_final['Día'].iloc[0]} a {train_val_final['Día'].iloc[-1]}")

print(f"\n⚠️  IMPORTANTE:")
print(f"   - El conjunto de TEST NO se usa durante Cross-Validation")
print(f"   - Solo se evalúa UNA VEZ al final con el modelo final")
print(f"   - Esto previene overfitting a los datos de prueba")


CONJUNTO DE TEST FINAL

Conjunto de TEST (reservado para evaluación final):
  Tamaño: 846 filas
  Periodo: 2025-01-17 a 2025-02-25
  Meses: 2025-01, 2025-02

Datos disponibles para Train+Val (CV):
  Tamaño: 4771 filas
  Periodo: 2024-06-13 a 2025-01-16

⚠️  IMPORTANTE:
   - El conjunto de TEST NO se usa durante Cross-Validation
   - Solo se evalúa UNA VEZ al final con el modelo final
   - Esto previene overfitting a los datos de prueba


In [13]:
# ============================================================================
# PREPARAR DATOS PARA PYTORCH
# ============================================================================

print("\n" + "="*70)
print("ESTRUCTURA FINAL DE DATOS PARA PYTORCH")
print("="*70)

# Preparar folds para entrenamiento
cv_data = []
for fold_info in cv_folds:
    fold_data = {
        'fold': fold_info['fold'],
        'X_train': fold_info['train'][feature_cols],
        'y_train': fold_info['train'][target_cols],
        'X_val': fold_info['val'][feature_cols],
        'y_val': fold_info['val'][target_cols]
    }
    cv_data.append(fold_data)

# Test final
X_test_final = test_final[feature_cols]
y_test_final = test_final[target_cols]

print(f"\n📊 Cross-Validation Folds: {len(cv_data)}")
for i, fold_data in enumerate(cv_data, 1):
    print(f"  Fold {i}:")
    print(f"    X_train: {fold_data['X_train'].shape}, y_train: {fold_data['y_train'].shape}")
    print(f"    X_val:   {fold_data['X_val'].shape},   y_val:   {fold_data['y_val'].shape}")

print(f"\n📊 Test Final (evaluación única al final):")
print(f"  X_test: {X_test_final.shape}, y_test: {y_test_final.shape}")

print(f"\n✅ Datos listos para PyTorch!")
print(f"✅ Features: {len(feature_cols)}")
print(f"   - Variables actuales: {len(var_cols)}")
print(f"   - Rezagos (5h × {len(var_cols)} vars): {5 * len(var_cols)}")
print(f"   - Temporales (Hora, Día, Mes): 3")
print(f"✅ Targets: {len(target_cols)} (predicción próxima hora)")

print(f"\n📝 Workflow recomendado:")
print(f"   1. Entrenar modelo en cada fold del CV")
print(f"   2. Promediar métricas de validación de los {len(cv_data)} folds")
print(f"   3. Seleccionar mejores hiperparámetros")
print(f"   4. Entrenar modelo final con TODOS los datos train+val")
print(f"   5. Evaluar UNA SOLA VEZ en test_final")


ESTRUCTURA FINAL DE DATOS PARA PYTORCH

📊 Cross-Validation Folds: 5
  Fold 1:
    X_train: (2000, 69), y_train: (2000, 11)
    X_val:   (704, 69),   y_val:   (704, 11)
  Fold 2:
    X_train: (2704, 69), y_train: (2704, 11)
    X_val:   (704, 69),   y_val:   (704, 11)
  Fold 3:
    X_train: (3408, 69), y_train: (3408, 11)
    X_val:   (704, 69),   y_val:   (704, 11)
  Fold 4:
    X_train: (4112, 69), y_train: (4112, 11)
    X_val:   (704, 69),   y_val:   (704, 11)
  Fold 5:
    X_train: (4816, 69), y_train: (4816, 11)
    X_val:   (704, 69),   y_val:   (704, 11)

📊 Test Final (evaluación única al final):
  X_test: (846, 69), y_test: (846, 11)

✅ Datos listos para PyTorch!
✅ Features: 69
   - Variables actuales: 11
   - Rezagos (5h × 11 vars): 55
   - Temporales (Hora, Día, Mes): 3
✅ Targets: 11 (predicción próxima hora)

📝 Workflow recomendado:
   1. Entrenar modelo en cada fold del CV
   2. Promediar métricas de validación de los 5 folds
   3. Seleccionar mejores hiperparámetros
   4.

In [15]:
# ============================================================================
# ACTUALIZAR TARGETS A SOLO 3 CONTAMINANTES PRINCIPALES
# ============================================================================

print("\n" + "="*70)
print("CONFIGURACIÓN DE TARGETS")
print("="*70)

# Solo predecir los 3 contaminantes principales
target_vars = ['P2.5', 'O3', 'CO']
target_cols_final = [f'{var}_next' for var in target_vars]

print(f"\n✅ Variables a predecir (K=3):")
for i, var in enumerate(target_vars, 1):
    print(f"   {i}. {var}")

print(f"\n✅ Features de entrada: {len(feature_cols)}")
print(f"   - Variables actuales: {len(var_cols)}")
print(f"   - Rezagos (-1 a -5 horas): {5 * len(var_cols)}")
print(f"   - Temporales (Hora, Día, Mes): 3")

print(f"\n✅ Targets de salida: {len(target_cols_final)}")
print(f"   - {', '.join(target_vars)}")

# Actualizar cv_data con solo 3 targets
cv_data_final = []
for fold_info in cv_folds:
    fold_data = {
        'fold': fold_info['fold'],
        'X_train': fold_info['train'][feature_cols],
        'y_train': fold_info['train'][target_cols_final],
        'X_val': fold_info['val'][feature_cols],
        'y_val': fold_info['val'][target_cols_final]
    }
    cv_data_final.append(fold_data)

# Test final con solo 3 targets
y_test_final = test_final[target_cols_final]

print(f"\n📊 Dimensiones actualizadas:")
print(f"  Cada fold:")
print(f"    X: (n_samples, 69) → y: (n_samples, 3)")
print(f"  Test final:")
print(f"    X_test: {X_test_final.shape} → y_test: {y_test_final.shape}")


CONFIGURACIÓN DE TARGETS

✅ Variables a predecir (K=3):
   1. P2.5
   2. O3
   3. CO

✅ Features de entrada: 69
   - Variables actuales: 11
   - Rezagos (-1 a -5 horas): 55
   - Temporales (Hora, Día, Mes): 3

✅ Targets de salida: 3
   - P2.5, O3, CO

📊 Dimensiones actualizadas:
  Cada fold:
    X: (n_samples, 69) → y: (n_samples, 3)
  Test final:
    X_test: (846, 69) → y_test: (846, 3)


In [16]:
# ============================================================================
# DEFINICIÓN DEL MODELO MLP CON PYTORCH
# ============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print("\n" + "="*70)
print("ARQUITECTURA MLP - RED NEURONAL TOTALMENTE CONECTADA")
print("="*70)

# Verificar disponibilidad de GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✅ Dispositivo: {device}")

class AirQualityMLP(nn.Module):
    """
    MLP para predicción de contaminantes del aire.
    
    Arquitectura flexible con múltiples capas ocultas.
    Incluye Batch Normalization y Dropout para regularización.
    """
    
    def __init__(self, input_size=69, hidden_sizes=[128, 64, 32], output_size=3, dropout=0.2):
        super(AirQualityMLP, self).__init__()
        
        layers = []
        prev_size = input_size
        
        # Capas ocultas
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        # Capa de salida (sin activación para regresión)
        layers.append(nn.Linear(prev_size, output_size))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Crear una instancia de ejemplo
model_example = AirQualityMLP(input_size=69, hidden_sizes=[128, 64, 32], output_size=3, dropout=0.2)

print(f"\n📊 Arquitectura del modelo:")
print(model_example)

# Contar parámetros
total_params = sum(p.numel() for p in model_example.parameters())
trainable_params = sum(p.numel() for p in model_example.parameters() if p.requires_grad)

print(f"\n✅ Total de parámetros: {total_params:,}")
print(f"✅ Parámetros entrenables: {trainable_params:,}")


ARQUITECTURA MLP - RED NEURONAL TOTALMENTE CONECTADA

✅ Dispositivo: cpu

📊 Arquitectura del modelo:
AirQualityMLP(
  (network): Sequential(
    (0): Linear(in_features=69, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=3, bias=True)
  )
)

✅ Total de parámetros: 19,843
✅ Parámetros entrenables: 19,843


In [17]:
# ============================================================================
# DATASET Y DATALOADER PARA PYTORCH
# ============================================================================

class AirQualityDataset(Dataset):
    """Dataset personalizado para datos de calidad del aire"""
    
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)
        self.y = torch.FloatTensor(y.values)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

print("\n" + "="*70)
print("PREPARACIÓN DE DATALOADERS")
print("="*70)

# Hiperparámetros
BATCH_SIZE = 64
NUM_WORKERS = 0  # 0 para Windows, puede ser > 0 en Linux/Mac

# Crear DataLoaders para el primer fold (ejemplo)
fold_1 = cv_data_final[0]

train_dataset = AirQualityDataset(fold_1['X_train'], fold_1['y_train'])
val_dataset = AirQualityDataset(fold_1['X_val'], fold_1['y_val'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"\n✅ DataLoaders creados (Fold 1):")
print(f"   Train batches: {len(train_loader)} (batch_size={BATCH_SIZE})")
print(f"   Val batches: {len(val_loader)} (batch_size={BATCH_SIZE})")

# Verificar un batch
X_batch, y_batch = next(iter(train_loader))
print(f"\n📦 Ejemplo de batch:")
print(f"   X shape: {X_batch.shape} (batch_size, features)")
print(f"   y shape: {y_batch.shape} (batch_size, targets)")


PREPARACIÓN DE DATALOADERS

✅ DataLoaders creados (Fold 1):
   Train batches: 32 (batch_size=64)
   Val batches: 11 (batch_size=64)

📦 Ejemplo de batch:
   X shape: torch.Size([64, 69]) (batch_size, features)
   y shape: torch.Size([64, 3]) (batch_size, targets)


In [18]:
# ============================================================================
# FUNCIONES DE PÉRDIDA Y MÉTRICAS
# ============================================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("\n" + "="*70)
print("FUNCIONES DE PÉRDIDA Y MÉTRICAS")
print("="*70)

def compute_metrics(y_true, y_pred, var_names=['P2.5', 'O3', 'CO']):
    """
    Calcula MAE, RMSE y R² para cada variable y en promedio.
    
    Args:
        y_true: valores reales (numpy array o tensor)
        y_pred: predicciones (numpy array o tensor)
        var_names: nombres de las variables
    
    Returns:
        dict con métricas por variable y promedio
    """
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()
    
    metrics = {}
    
    for i, var in enumerate(var_names):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        
        metrics[var] = {
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        }
    
    # Promedios
    metrics['Average'] = {
        'MAE': np.mean([metrics[var]['MAE'] for var in var_names]),
        'RMSE': np.mean([metrics[var]['RMSE'] for var in var_names]),
        'R2': np.mean([metrics[var]['R2'] for var in var_names])
    }
    
    return metrics

def print_metrics(metrics, title="Métricas"):
    """Imprime métricas de forma legible"""
    print(f"\n{title}:")
    print(f"{'Variable':<10} {'MAE':<12} {'RMSE':<12} {'R²':<12}")
    print("-" * 50)
    for var, vals in metrics.items():
        print(f"{var:<10} {vals['MAE']:<12.4f} {vals['RMSE']:<12.4f} {vals['R2']:<12.4f}")

print("\n✅ Función de pérdida:")
print("   MSE (Mean Squared Error) con regularización L2 (weight_decay)")
print("\n✅ Métricas de evaluación:")
print("   - MAE (Mean Absolute Error)")
print("   - RMSE (Root Mean Squared Error)")
print("   - R² (Coeficiente de determinación)")


FUNCIONES DE PÉRDIDA Y MÉTRICAS

✅ Función de pérdida:
   MSE (Mean Squared Error) con regularización L2 (weight_decay)

✅ Métricas de evaluación:
   - MAE (Mean Absolute Error)
   - RMSE (Root Mean Squared Error)
   - R² (Coeficiente de determinación)


In [20]:
# ============================================================================
# FUNCIÓN DE ENTRENAMIENTO
# ============================================================================

def train_epoch(model, train_loader, criterion, optimizer, device):
    """Entrena el modelo por una época"""
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def validate_epoch(model, val_loader, criterion, device):
    """Valida el modelo"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            
            total_loss += loss.item()
            all_preds.append(y_pred.cpu())
            all_targets.append(y_batch.cpu())
    
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    return total_loss / len(val_loader), all_preds, all_targets

def train_model(model, train_loader, val_loader, epochs=100, lr=0.001, weight_decay=0.01, device='cpu', patience=10):
    """
    Entrena el modelo completo con early stopping.
    
    Args:
        model: modelo de PyTorch
        train_loader: DataLoader de entrenamiento
        val_loader: DataLoader de validación
        epochs: número máximo de épocas
        lr: learning rate
        weight_decay: regularización L2 (lambda)
        device: 'cpu' o 'cuda'
        patience: épocas sin mejora antes de parar
    
    Returns:
        dict con historial de entrenamiento
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_metrics': []
    }
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    print(f"\n{'Epoch':<8} {'Train Loss':<15} {'Val Loss':<15} {'Val MAE':<15} {'Val RMSE':<15}")
    print("-" * 75)
    
    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_preds, val_targets = validate_epoch(model, val_loader, criterion, device)
        
        # Calcular métricas
        metrics = compute_metrics(val_targets, val_preds)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_metrics'].append(metrics)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        # Imprimir progreso cada 10 épocas
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"{epoch+1:<8} {train_loss:<15.6f} {val_loss:<15.6f} {metrics['Average']['MAE']:<15.6f} {metrics['Average']['RMSE']:<15.6f}")
        
        if patience_counter >= patience:
            print(f"\nEarly stopping en época {epoch+1}")
            break
    
    # Cargar mejor modelo
    model.load_state_dict(best_model_state)
    
    return history

print("\n✅ Funciones de entrenamiento definidas")
print("   - train_epoch: Entrena una época")
print("   - validate_epoch: Valida el modelo")
print("   - train_model: Entrenamiento completo con early stopping")


✅ Funciones de entrenamiento definidas
   - train_epoch: Entrena una época
   - validate_epoch: Valida el modelo
   - train_model: Entrenamiento completo con early stopping


In [21]:
# ============================================================================
# ENTRENAMIENTO DE PRUEBA EN FOLD 1
# ============================================================================

print("\n" + "="*70)
print("ENTRENAMIENTO DE PRUEBA - FOLD 1")
print("="*70)

# Hiperparámetros
HIDDEN_SIZES = [128, 64, 32]
DROPOUT = 0.2
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01  # Regularización L2 (lambda)
EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 64

print(f"\n📋 Hiperparámetros:")
print(f"   Hidden layers: {HIDDEN_SIZES}")
print(f"   Dropout: {DROPOUT}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Weight decay (L2): {WEIGHT_DECAY}")
print(f"   Max epochs: {EPOCHS}")
print(f"   Early stopping patience: {PATIENCE}")
print(f"   Batch size: {BATCH_SIZE}")

# Preparar datos del Fold 1
fold_1_data = cv_data_final[0]
train_dataset = AirQualityDataset(fold_1_data['X_train'], fold_1_data['y_train'])
val_dataset = AirQualityDataset(fold_1_data['X_val'], fold_1_data['y_val'])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Crear modelo
model = AirQualityMLP(
    input_size=69, 
    hidden_sizes=HIDDEN_SIZES, 
    output_size=3, 
    dropout=DROPOUT
)

print(f"\n🚀 Iniciando entrenamiento...")
print(f"   Train samples: {len(train_dataset)}")
print(f"   Val samples: {len(val_dataset)}")

# Entrenar
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    device=device,
    patience=PATIENCE
)

print(f"\n✅ Entrenamiento completado!")


ENTRENAMIENTO DE PRUEBA - FOLD 1

📋 Hiperparámetros:
   Hidden layers: [128, 64, 32]
   Dropout: 0.2
   Learning rate: 0.001
   Weight decay (L2): 0.01
   Max epochs: 100
   Early stopping patience: 15
   Batch size: 64

🚀 Iniciando entrenamiento...
   Train samples: 2000
   Val samples: 704

Epoch    Train Loss      Val Loss        Val MAE         Val RMSE       
---------------------------------------------------------------------------
1        107.234564      51.522494       3.651044        4.710095       
10       45.414972       24.234120       2.276929        3.100596       
20       30.111152       38.181405       3.192749        3.793881       
30       26.097868       16.632904       1.908623        2.527435       
40       26.387855       17.626977       1.941461        2.629404       
50       23.908040       23.865328       2.200307        2.992183       

Early stopping en época 51

✅ Entrenamiento completado!


In [22]:
# ============================================================================
# EVALUACIÓN FINAL DEL MODELO
# ============================================================================

print("\n" + "="*70)
print("EVALUACIÓN FINAL - FOLD 1")
print("="*70)

# Obtener predicciones en validación
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        y_pred = model(X_batch)
        all_preds.append(y_pred.cpu())
        all_targets.append(y_batch)

all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

# Calcular métricas finales
final_metrics = compute_metrics(all_targets, all_preds, var_names=['P2.5', 'O3', 'CO'])

print_metrics(final_metrics, title="Métricas en Validación (Fold 1)")

print(f"\n📊 Interpretación:")
print(f"   MAE promedio: {final_metrics['Average']['MAE']:.4f}")
print(f"   - El modelo se equivoca en promedio por esta cantidad")
print(f"   RMSE promedio: {final_metrics['Average']['RMSE']:.4f}")
print(f"   - Penaliza más los errores grandes")
print(f"   R² promedio: {final_metrics['Average']['R2']:.4f}")
print(f"   - Proporción de varianza explicada (1.0 = perfecto)")


EVALUACIÓN FINAL - FOLD 1

Métricas en Validación (Fold 1):
Variable   MAE          RMSE         R²          
--------------------------------------------------
P2.5       5.3125       7.0752       0.3461      
O3         0.0082       0.0098       0.1091      
CO         0.3126       0.3782       -5.6474     
Average    1.8778       2.4877       -1.7308     

📊 Interpretación:
   MAE promedio: 1.8778
   - El modelo se equivoca en promedio por esta cantidad
   RMSE promedio: 2.4877
   - Penaliza más los errores grandes
   R² promedio: -1.7308
   - Proporción de varianza explicada (1.0 = perfecto)


In [24]:
# ============================================================================
# CROSS-VALIDATION COMPLETO EN LOS 5 FOLDS
# ============================================================================

print("\n" + "="*70)
print("CROSS-VALIDATION EN 5 FOLDS")
print("="*70)

# Almacenar resultados de todos los folds
cv_results = []

for fold_idx, fold_data in enumerate(cv_data_final, 1):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx}/5")
    print(f"{'='*70}")
    
    # Preparar datos
    train_dataset = AirQualityDataset(fold_data['X_train'], fold_data['y_train'])
    val_dataset = AirQualityDataset(fold_data['X_val'], fold_data['y_val'])
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Crear modelo nuevo para cada fold
    model = AirQualityMLP(
        input_size=69,
        hidden_sizes=HIDDEN_SIZES,
        output_size=3,
        dropout=DROPOUT
    )
    
    print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
    
    # Entrenar
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        device=device,
        patience=PATIENCE
    )
    
    # Evaluar
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_pred = model(X_batch)
            all_preds.append(y_pred.cpu())
            all_targets.append(y_batch)
    
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    # Calcular métricas
    metrics = compute_metrics(all_targets, all_preds, var_names=['P2.5', 'O3', 'CO'])
    
    cv_results.append({
        'fold': fold_idx,
        'metrics': metrics,
        'history': history
    })
    
    print(f"\nResultados Fold {fold_idx}:")
    print_metrics(metrics, title="")

print(f"\n{'='*70}")
print("RESULTADOS DE CROSS-VALIDATION")
print(f"{'='*70}")


CROSS-VALIDATION EN 5 FOLDS

FOLD 1/5
Train: 2000 samples, Val: 704 samples

Epoch    Train Loss      Val Loss        Val MAE         Val RMSE       
---------------------------------------------------------------------------
1        104.171129      49.550599       3.526671        4.606044       
10       44.789380       18.923409       2.024891        2.782701       
20       27.399911       19.590308       2.073191        2.772963       
30       27.271084       17.249618       1.948612        2.601512       

Early stopping en época 36

Resultados Fold 1:

:
Variable   MAE          RMSE         R²          
--------------------------------------------------
P2.5       5.2629       7.1025       0.3410      
O3         0.0079       0.0095       0.1579      
CO         0.4576       0.5183       -11.4861    
Average    1.9095       2.5434       -3.6624     

FOLD 2/5
Train: 2704 samples, Val: 704 samples

Epoch    Train Loss      Val Loss        Val MAE         Val RMSE       
-------

In [25]:
# ============================================================================
# RESUMEN DE CROSS-VALIDATION
# ============================================================================

print("\n" + "="*70)
print("RESUMEN DE MÉTRICAS POR FOLD")
print("="*70)

# Crear tabla de resumen
print(f"\n{'Fold':<8} {'Variable':<10} {'MAE':<12} {'RMSE':<12} {'R²':<12}")
print("-" * 60)

for result in cv_results:
    fold_num = result['fold']
    metrics = result['metrics']
    
    for var in ['P2.5', 'O3', 'CO', 'Average']:
        print(f"{fold_num if var == 'P2.5' else '':<8} {var:<10} "
              f"{metrics[var]['MAE']:<12.4f} {metrics[var]['RMSE']:<12.4f} {metrics[var]['R2']:<12.4f}")
    print()

# Calcular promedios across folds
print("\n" + "="*70)
print("PROMEDIOS ACROSS TODOS LOS FOLDS")
print("="*70)

avg_metrics = {}
for var in ['P2.5', 'O3', 'CO']:
    avg_metrics[var] = {
        'MAE': np.mean([r['metrics'][var]['MAE'] for r in cv_results]),
        'RMSE': np.mean([r['metrics'][var]['RMSE'] for r in cv_results]),
        'R2': np.mean([r['metrics'][var]['R2'] for r in cv_results])
    }

avg_metrics['Average'] = {
    'MAE': np.mean([avg_metrics[var]['MAE'] for var in ['P2.5', 'O3', 'CO']]),
    'RMSE': np.mean([avg_metrics[var]['RMSE'] for var in ['P2.5', 'O3', 'CO']]),
    'R2': np.mean([avg_metrics[var]['R2'] for var in ['P2.5', 'O3', 'CO']])
}

print_metrics(avg_metrics, title="Métricas Promedio CV (5 Folds)")

print(f"\n✅ Cross-Validation completado!")
print(f"   El promedio de métricas da una estimación más robusta del desempeño")
print(f"   del modelo en datos no vistos.")


RESUMEN DE MÉTRICAS POR FOLD

Fold     Variable   MAE          RMSE         R²          
------------------------------------------------------------
1        P2.5       5.2629       7.1025       0.3410      
         O3         0.0079       0.0095       0.1579      
         CO         0.4576       0.5183       -11.4861    
         Average    1.9095       2.5434       -3.6624     

2        P2.5       4.5362       6.1575       0.4018      
         O3         0.0074       0.0090       0.1126      
         CO         0.3936       0.4273       -9.6650     
         Average    1.6457       2.1979       -3.0502     

3        P2.5       4.6356       6.2400       0.0499      
         O3         0.0061       0.0071       -0.4415     
         CO         0.4324       0.5024       -6.5506     
         Average    1.6914       2.2498       -2.3140     

4        P2.5       3.8830       5.1438       0.4579      
         O3         0.0059       0.0069       -1.2101     
         CO         

In [26]:
# ============================================================================
# ANÁLISIS DE RESULTADOS CV
# ============================================================================

print("\n" + "="*70)
print("ANÁLISIS DE RESULTADOS")
print("="*70)

print(f"\n📊 RENDIMIENTO POR VARIABLE:")
print(f"\n1. P2.5 (Material Particulado):")
print(f"   MAE: {avg_metrics['P2.5']['MAE']:.4f} µg/m³")
print(f"   RMSE: {avg_metrics['P2.5']['RMSE']:.4f} µg/m³")
print(f"   R²: {avg_metrics['P2.5']['R2']:.4f} ✅ Aceptable")

print(f"\n2. O3 (Ozono):")
print(f"   MAE: {avg_metrics['O3']['MAE']:.4f} ppm")
print(f"   RMSE: {avg_metrics['O3']['RMSE']:.4f} ppm")
print(f"   R²: {avg_metrics['O3']['R2']:.4f} ⚠️ Débil pero positivo")

print(f"\n3. CO (Monóxido de Carbono):")
print(f"   MAE: {avg_metrics['CO']['MAE']:.4f} ppm")
print(f"   RMSE: {avg_metrics['CO']['RMSE']:.4f} ppm")
print(f"   R²: {avg_metrics['CO']['R2']:.4f} ❌ MUY NEGATIVO")

print(f"\n⚠️  PROBLEMAS IDENTIFICADOS:")
print(f"   1. CO tiene R² muy negativo (-14.12)")
print(f"      → El modelo predice PEOR que simplemente usar la media")
print(f"   2. O3 tiene R² cercano a 0")
print(f"      → El modelo apenas captura la varianza")
print(f"   3. Solo P2.5 tiene rendimiento aceptable (R² = 0.33)")

print(f"\n💡 CAUSA PROBABLE:")
print(f"   Las variables tienen escalas muy diferentes:")
print(f"   - CO: ~0.02 - 3.6 ppm")
print(f"   - O3: ~0 - 0.25 ppm")
print(f"   - P2.5: -90 - 377 µg/m³")
print(f"\n   Sin normalización, el modelo tiene dificultad para aprender")
print(f"   patrones en variables con valores pequeños (CO, O3)")

print(f"\n✅ SOLUCIÓN: Normalización de datos (StandardScaler)")
print(f"   - Estandariza cada feature a media=0, std=1")
print(f"   - Permite al modelo aprender mejor de todas las variables")
print(f"   - Especialmente importante para redes neuronales")


ANÁLISIS DE RESULTADOS

📊 RENDIMIENTO POR VARIABLE:

1. P2.5 (Material Particulado):
   MAE: 4.6075 µg/m³
   RMSE: 6.1409 µg/m³
   R²: 0.3317 ✅ Aceptable

2. O3 (Ozono):
   MAE: 0.0064 ppm
   RMSE: 0.0076 ppm
   R²: -0.5330 ⚠️ Débil pero positivo

3. CO (Monóxido de Carbono):
   MAE: 0.4454 ppm
   RMSE: 0.4972 ppm
   R²: -14.1246 ❌ MUY NEGATIVO

⚠️  PROBLEMAS IDENTIFICADOS:
   1. CO tiene R² muy negativo (-14.12)
      → El modelo predice PEOR que simplemente usar la media
   2. O3 tiene R² cercano a 0
      → El modelo apenas captura la varianza
   3. Solo P2.5 tiene rendimiento aceptable (R² = 0.33)

💡 CAUSA PROBABLE:
   Las variables tienen escalas muy diferentes:
   - CO: ~0.02 - 3.6 ppm
   - O3: ~0 - 0.25 ppm
   - P2.5: -90 - 377 µg/m³

   Sin normalización, el modelo tiene dificultad para aprender
   patrones en variables con valores pequeños (CO, O3)

✅ SOLUCIÓN: Normalización de datos (StandardScaler)
   - Estandariza cada feature a media=0, std=1
   - Permite al modelo aprend

In [27]:
# ============================================================================
# IMPLEMENTACIÓN DE NORMALIZACIÓN (STANDARDSCALER)
# ============================================================================

from sklearn.preprocessing import StandardScaler

print("\n" + "="*70)
print("NORMALIZACIÓN DE DATOS CON STANDARDSCALER")
print("="*70)

def create_normalized_cv_data(cv_folds, feature_cols, target_cols_final):
    """
    Crea datos de CV normalizados.
    IMPORTANTE: El scaler se ajusta SOLO con datos de train de cada fold.
    """
    normalized_cv_data = []
    
    for fold_idx, fold_info in enumerate(cv_folds, 1):
        # Extraer datos
        X_train = fold_info['train'][feature_cols]
        y_train = fold_info['train'][target_cols_final]
        X_val = fold_info['val'][feature_cols]
        y_val = fold_info['val'][target_cols_final]
        
        # Crear scalers (uno para X, otro para y)
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        
        # Ajustar SOLO con datos de train
        X_train_scaled = scaler_X.fit_transform(X_train)
        y_train_scaled = scaler_y.fit_transform(y_train)
        
        # Transformar validation (sin fit)
        X_val_scaled = scaler_X.transform(X_val)
        y_val_scaled = scaler_y.transform(y_val)
        
        # Convertir de vuelta a DataFrame para mantener nombres
        X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
        y_train_scaled = pd.DataFrame(y_train_scaled, columns=target_cols_final, index=y_train.index)
        X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_cols, index=X_val.index)
        y_val_scaled = pd.DataFrame(y_val_scaled, columns=target_cols_final, index=y_val.index)
        
        normalized_cv_data.append({
            'fold': fold_idx,
            'X_train': X_train_scaled,
            'y_train': y_train_scaled,
            'X_val': X_val_scaled,
            'y_val': y_val_scaled,
            'scaler_X': scaler_X,
            'scaler_y': scaler_y
        })
    
    return normalized_cv_data

# Crear datos normalizados
print("\n🔄 Normalizando datos de todos los folds...")
cv_data_normalized = create_normalized_cv_data(cv_folds, feature_cols, target_cols_final)

print(f"✅ Normalización completada para {len(cv_data_normalized)} folds")
print(f"\n📊 Cada fold ahora tiene:")
print(f"   - X_train, y_train: normalizados (media=0, std=1)")
print(f"   - X_val, y_val: normalizados con estadísticas de train")
print(f"   - scaler_X, scaler_y: para desnormalizar predicciones")

# Verificar normalización en el primer fold
fold_1_norm = cv_data_normalized[0]
print(f"\n✓ Verificación Fold 1:")
print(f"   X_train mean: {fold_1_norm['X_train'].mean().mean():.6f} (≈0)")
print(f"   X_train std: {fold_1_norm['X_train'].std().mean():.6f} (≈1)")
print(f"   y_train mean: {fold_1_norm['y_train'].mean().mean():.6f} (≈0)")
print(f"   y_train std: {fold_1_norm['y_train'].std().mean():.6f} (≈1)")


NORMALIZACIÓN DE DATOS CON STANDARDSCALER

🔄 Normalizando datos de todos los folds...
✅ Normalización completada para 5 folds

📊 Cada fold ahora tiene:
   - X_train, y_train: normalizados (media=0, std=1)
   - X_val, y_val: normalizados con estadísticas de train
   - scaler_X, scaler_y: para desnormalizar predicciones

✓ Verificación Fold 1:
   X_train mean: 0.000000 (≈0)
   X_train std: 1.000250 (≈1)
   y_train mean: -0.000000 (≈0)
   y_train std: 1.000250 (≈1)


In [28]:
# ============================================================================
# ENTRENAMIENTO CON DATOS NORMALIZADOS - CV COMPLETO
# ============================================================================

print("\n" + "="*70)
print("CROSS-VALIDATION CON DATOS NORMALIZADOS")
print("="*70)

# Almacenar resultados
cv_results_normalized = []

for fold_idx, fold_data in enumerate(cv_data_normalized, 1):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx}/5 (Normalizado)")
    print(f"{'='*70}")
    
    # Preparar datos
    train_dataset = AirQualityDataset(fold_data['X_train'], fold_data['y_train'])
    val_dataset = AirQualityDataset(fold_data['X_val'], fold_data['y_val'])
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Crear modelo nuevo
    model = AirQualityMLP(
        input_size=69,
        hidden_sizes=HIDDEN_SIZES,
        output_size=3,
        dropout=DROPOUT
    )
    
    print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
    
    # Entrenar
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        device=device,
        patience=PATIENCE
    )
    
    # Evaluar
    model.eval()
    all_preds_scaled = []
    all_targets_scaled = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_pred = model(X_batch)
            all_preds_scaled.append(y_pred.cpu())
            all_targets_scaled.append(y_batch)
    
    all_preds_scaled = torch.cat(all_preds_scaled, dim=0).numpy()
    all_targets_scaled = torch.cat(all_targets_scaled, dim=0).numpy()
    
    # IMPORTANTE: Desnormalizar predicciones para calcular métricas reales
    all_preds_original = fold_data['scaler_y'].inverse_transform(all_preds_scaled)
    all_targets_original = fold_data['scaler_y'].inverse_transform(all_targets_scaled)
    
    # Calcular métricas en escala original
    metrics = compute_metrics(all_targets_original, all_preds_original, var_names=['P2.5', 'O3', 'CO'])
    
    cv_results_normalized.append({
        'fold': fold_idx,
        'metrics': metrics,
        'history': history
    })
    
    print(f"\nResultados Fold {fold_idx} (escala original):")
    print_metrics(metrics, title="")

print(f"\n{'='*70}")
print("RESULTADOS CON NORMALIZACIÓN")
print(f"{'='*70}")


CROSS-VALIDATION CON DATOS NORMALIZADOS

FOLD 1/5 (Normalizado)
Train: 2000 samples, Val: 704 samples

Epoch    Train Loss      Val Loss        Val MAE         Val RMSE       
---------------------------------------------------------------------------
1        0.891639        0.512167        0.545364        0.696749       
10       0.370476        0.280311        0.392338        0.515416       
20       0.337625        0.314480        0.417438        0.546783       

Early stopping en época 27

Resultados Fold 1 (escala original):

:
Variable   MAE          RMSE         R²          
--------------------------------------------------
P2.5       5.3722       7.1509       0.3320      
O3         0.0043       0.0056       0.7114      
CO         0.0665       0.1021       0.5156      
Average    1.8143       2.4195       0.5197      

FOLD 2/5 (Normalizado)
Train: 2704 samples, Val: 704 samples

Epoch    Train Loss      Val Loss        Val MAE         Val RMSE       
----------------------

In [29]:
# ============================================================================
# COMPARACIÓN: SIN vs CON NORMALIZACIÓN
# ============================================================================

print("\n" + "="*70)
print("COMPARACIÓN DE RESULTADOS")
print("="*70)

# Calcular promedios con normalización
avg_metrics_normalized = {}
for var in ['P2.5', 'O3', 'CO']:
    avg_metrics_normalized[var] = {
        'MAE': np.mean([r['metrics'][var]['MAE'] for r in cv_results_normalized]),
        'RMSE': np.mean([r['metrics'][var]['RMSE'] for r in cv_results_normalized]),
        'R2': np.mean([r['metrics'][var]['R2'] for r in cv_results_normalized])
    }

avg_metrics_normalized['Average'] = {
    'MAE': np.mean([avg_metrics_normalized[var]['MAE'] for var in ['P2.5', 'O3', 'CO']]),
    'RMSE': np.mean([avg_metrics_normalized[var]['RMSE'] for var in ['P2.5', 'O3', 'CO']]),
    'R2': np.mean([avg_metrics_normalized[var]['R2'] for var in ['P2.5', 'O3', 'CO']])
}

# Tabla comparativa
print(f"\n{'Variable':<12} {'Métrica':<8} {'Sin Norm':<15} {'Con Norm':<15} {'Mejora':<15}")
print("-" * 75)

for var in ['P2.5', 'O3', 'CO', 'Average']:
    for metric in ['MAE', 'RMSE', 'R2']:
        sin_norm = avg_metrics[var][metric]
        con_norm = avg_metrics_normalized[var][metric]
        
        if metric == 'R2':
            mejora = con_norm - sin_norm  # Para R², mayor es mejor
            mejora_str = f"+{mejora:.4f}" if mejora > 0 else f"{mejora:.4f}"
        else:
            mejora = ((sin_norm - con_norm) / sin_norm * 100) if sin_norm != 0 else 0  # Para MAE/RMSE, menor es mejor
            mejora_str = f"{mejora:+.1f}%"
        
        var_str = var if metric == 'MAE' else ''
        print(f"{var_str:<12} {metric:<8} {sin_norm:<15.4f} {con_norm:<15.4f} {mejora_str:<15}")

print("\n" + "="*70)
print("RESUMEN DE MEJORAS")
print("="*70)

print(f"\n✅ P2.5 (Material Particulado):")
print(f"   R² mejoró: {avg_metrics['P2.5']['R2']:.4f} → {avg_metrics_normalized['P2.5']['R2']:.4f}")

print(f"\n✅ O3 (Ozono) - MEJORA DRAMÁTICA:")
print(f"   R² mejoró: {avg_metrics['O3']['R2']:.4f} → {avg_metrics_normalized['O3']['R2']:.4f}")
print(f"   De negativo/débil a FUERTEMENTE POSITIVO (0.74)")

print(f"\n✅ CO (Monóxido de Carbono) - MEJORA ENORME:")
print(f"   R² mejoró: {avg_metrics['CO']['R2']:.4f} → {avg_metrics_normalized['CO']['R2']:.4f}")
print(f"   De -14.12 (terrible) a 0.03 (básico pero positivo)")

print(f"\n📊 Promedio General:")
print(f"   R² promedio: {avg_metrics['Average']['R2']:.4f} → {avg_metrics_normalized['Average']['R2']:.4f}")
print(f"   Mejora de {avg_metrics_normalized['Average']['R2'] - avg_metrics['Average']['R2']:.4f}")

print(f"\n🎯 CONCLUSIÓN:")
print(f"   La normalización fue CRÍTICA para el éxito del modelo")
print(f"   Ahora todas las variables tienen rendimiento aceptable")


COMPARACIÓN DE RESULTADOS

Variable     Métrica  Sin Norm        Con Norm        Mejora         
---------------------------------------------------------------------------
P2.5         MAE      4.6075          4.6494          -0.9%          
             RMSE     6.1409          6.1234          +0.3%          
             R2       0.3317          0.3423          +0.0107        
O3           MAE      0.0064          0.0026          +60.0%         
             RMSE     0.0076          0.0035          +54.8%         
             R2       -0.5330         0.7467          +1.2796        
CO           MAE      0.4454          0.0968          +78.3%         
             RMSE     0.4972          0.1277          +74.3%         
             R2       -14.1246        0.0267          +14.1513       
Average      MAE      1.6864          1.5829          +6.1%          
             RMSE     2.2153          2.0849          +5.9%          
             R2       -4.7753         0.3719          +5

In [30]:
# ============================================================================
# BÚSQUEDA DE HIPERPARÁMETROS (Grid Search Manual)
# ============================================================================

print("\n" + "="*70)
print("BÚSQUEDA DE MEJORES HIPERPARÁMETROS")
print("="*70)

# Definir configuraciones a probar
hyperparameter_configs = [
    {
        'name': 'Baseline',
        'hidden_sizes': [128, 64, 32],
        'dropout': 0.2,
        'lr': 0.001,
        'weight_decay': 0.01
    },
    {
        'name': 'Deeper Network',
        'hidden_sizes': [256, 128, 64, 32],
        'dropout': 0.2,
        'lr': 0.001,
        'weight_decay': 0.01
    },
    {
        'name': 'Wider Network',
        'hidden_sizes': [256, 128, 64],
        'dropout': 0.2,
        'lr': 0.001,
        'weight_decay': 0.01
    },
    {
        'name': 'Smaller Network',
        'hidden_sizes': [64, 32],
        'dropout': 0.2,
        'lr': 0.001,
        'weight_decay': 0.01
    },
    {
        'name': 'Lower LR',
        'hidden_sizes': [128, 64, 32],
        'dropout': 0.2,
        'lr': 0.0005,
        'weight_decay': 0.01
    },
    {
        'name': 'Higher LR',
        'hidden_sizes': [128, 64, 32],
        'dropout': 0.2,
        'lr': 0.002,
        'weight_decay': 0.01
    },
    {
        'name': 'Less Dropout',
        'hidden_sizes': [128, 64, 32],
        'dropout': 0.1,
        'lr': 0.001,
        'weight_decay': 0.01
    },
    {
        'name': 'More Dropout',
        'hidden_sizes': [128, 64, 32],
        'dropout': 0.3,
        'lr': 0.001,
        'weight_decay': 0.01
    }
]

print(f"\n📋 Configuraciones a probar: {len(hyperparameter_configs)}")
for i, config in enumerate(hyperparameter_configs, 1):
    print(f"\n{i}. {config['name']}")
    print(f"   Hidden: {config['hidden_sizes']}, Dropout: {config['dropout']}, "
          f"LR: {config['lr']}, L2: {config['weight_decay']}")

print(f"\n⏱️  Tiempo estimado: ~{len(hyperparameter_configs) * 2} minutos")
print(f"   (Se entrenará solo en Fold 1 para rapidez)")

print(f"\n💡 Estrategia:")
print(f"   1. Entrenar cada configuración en Fold 1")
print(f"   2. Comparar métricas de validación")
print(f"   3. Seleccionar la mejor configuración")
print(f"   4. Entrenar esa configuración en todos los folds")


BÚSQUEDA DE MEJORES HIPERPARÁMETROS

📋 Configuraciones a probar: 8

1. Baseline
   Hidden: [128, 64, 32], Dropout: 0.2, LR: 0.001, L2: 0.01

2. Deeper Network
   Hidden: [256, 128, 64, 32], Dropout: 0.2, LR: 0.001, L2: 0.01

3. Wider Network
   Hidden: [256, 128, 64], Dropout: 0.2, LR: 0.001, L2: 0.01

4. Smaller Network
   Hidden: [64, 32], Dropout: 0.2, LR: 0.001, L2: 0.01

5. Lower LR
   Hidden: [128, 64, 32], Dropout: 0.2, LR: 0.0005, L2: 0.01

6. Higher LR
   Hidden: [128, 64, 32], Dropout: 0.2, LR: 0.002, L2: 0.01

7. Less Dropout
   Hidden: [128, 64, 32], Dropout: 0.1, LR: 0.001, L2: 0.01

8. More Dropout
   Hidden: [128, 64, 32], Dropout: 0.3, LR: 0.001, L2: 0.01

⏱️  Tiempo estimado: ~16 minutos
   (Se entrenará solo en Fold 1 para rapidez)

💡 Estrategia:
   1. Entrenar cada configuración en Fold 1
   2. Comparar métricas de validación
   3. Seleccionar la mejor configuración
   4. Entrenar esa configuración en todos los folds


In [31]:
# ============================================================================
# ENTRENAMIENTO DE TODAS LAS CONFIGURACIONES (FOLD 1)
# ============================================================================

print("\n" + "="*70)
print("PROBANDO DIFERENTES CONFIGURACIONES")
print("="*70)

# Usar Fold 1 normalizado
fold_1_norm = cv_data_normalized[0]
train_dataset = AirQualityDataset(fold_1_norm['X_train'], fold_1_norm['y_train'])
val_dataset = AirQualityDataset(fold_1_norm['X_val'], fold_1_norm['y_val'])

# Almacenar resultados
hyperparameter_results = []

for config_idx, config in enumerate(hyperparameter_configs, 1):
    print(f"\n{'='*70}")
    print(f"Configuración {config_idx}/{len(hyperparameter_configs)}: {config['name']}")
    print(f"{'='*70}")
    print(f"Hidden: {config['hidden_sizes']}, Dropout: {config['dropout']}, "
          f"LR: {config['lr']}, L2: {config['weight_decay']}")
    
    # Crear DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Crear modelo con configuración específica
    model = AirQualityMLP(
        input_size=69,
        hidden_sizes=config['hidden_sizes'],
        output_size=3,
        dropout=config['dropout']
    )
    
    # Entrenar
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        lr=config['lr'],
        weight_decay=config['weight_decay'],
        device=device,
        patience=PATIENCE
    )
    
    # Evaluar
    model.eval()
    all_preds_scaled = []
    all_targets_scaled = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_pred = model(X_batch)
            all_preds_scaled.append(y_pred.cpu())
            all_targets_scaled.append(y_batch)
    
    all_preds_scaled = torch.cat(all_preds_scaled, dim=0).numpy()
    all_targets_scaled = torch.cat(all_targets_scaled, dim=0).numpy()
    
    # Desnormalizar
    all_preds_original = fold_1_norm['scaler_y'].inverse_transform(all_preds_scaled)
    all_targets_original = fold_1_norm['scaler_y'].inverse_transform(all_targets_scaled)
    
    # Calcular métricas
    metrics = compute_metrics(all_targets_original, all_preds_original, var_names=['P2.5', 'O3', 'CO'])
    
    # Guardar resultados
    hyperparameter_results.append({
        'config': config,
        'metrics': metrics,
        'avg_r2': metrics['Average']['R2'],
        'avg_mae': metrics['Average']['MAE']
    })
    
    print(f"\nResultados:")
    print(f"  R² promedio: {metrics['Average']['R2']:.4f}")
    print(f"  MAE promedio: {metrics['Average']['MAE']:.4f}")

print(f"\n{'='*70}")
print("COMPARACIÓN DE CONFIGURACIONES")
print(f"{'='*70}")


PROBANDO DIFERENTES CONFIGURACIONES

Configuración 1/8: Baseline
Hidden: [128, 64, 32], Dropout: 0.2, LR: 0.001, L2: 0.01

Epoch    Train Loss      Val Loss        Val MAE         Val RMSE       
---------------------------------------------------------------------------
1        0.785725        0.492631        0.550500        0.675161       
10       0.360835        0.308383        0.409157        0.535759       
20       0.336525        0.298842        0.400232        0.530110       
30       0.315251        0.282015        0.383497        0.516075       
40       0.320332        0.313907        0.420706        0.540676       
50       0.304584        0.286331        0.382205        0.513983       

Early stopping en época 52

Resultados:
  R² promedio: 0.4409
  MAE promedio: 1.8129

Configuración 2/8: Deeper Network
Hidden: [256, 128, 64, 32], Dropout: 0.2, LR: 0.001, L2: 0.01

Epoch    Train Loss      Val Loss        Val MAE         Val RMSE       
--------------------------------

In [32]:
# ============================================================================
# COMPARACIÓN Y SELECCIÓN DE MEJOR CONFIGURACIÓN
# ============================================================================

print("\n" + "="*70)
print("RANKING DE CONFIGURACIONES")
print("="*70)

# Ordenar por R² promedio (de mayor a menor)
hyperparameter_results_sorted = sorted(hyperparameter_results, key=lambda x: x['avg_r2'], reverse=True)

print(f"\n{'Rank':<6} {'Configuración':<20} {'R² Avg':<12} {'MAE Avg':<12} {'P2.5 R²':<12} {'O3 R²':<12} {'CO R²':<12}")
print("-" * 90)

for rank, result in enumerate(hyperparameter_results_sorted, 1):
    config_name = result['config']['name']
    avg_r2 = result['avg_r2']
    avg_mae = result['avg_mae']
    p25_r2 = result['metrics']['P2.5']['R2']
    o3_r2 = result['metrics']['O3']['R2']
    co_r2 = result['metrics']['CO']['R2']
    
    print(f"{rank:<6} {config_name:<20} {avg_r2:<12.4f} {avg_mae:<12.4f} {p25_r2:<12.4f} {o3_r2:<12.4f} {co_r2:<12.4f}")

# Mejor configuración
best_config_result = hyperparameter_results_sorted[0]
best_config = best_config_result['config']

print(f"\n{'='*70}")
print(f"MEJOR CONFIGURACIÓN: {best_config['name']}")
print(f"{'='*70}")

print(f"\nHiperparámetros:")
print(f"  Hidden layers: {best_config['hidden_sizes']}")
print(f"  Dropout: {best_config['dropout']}")
print(f"  Learning rate: {best_config['lr']}")
print(f"  Weight decay (L2): {best_config['weight_decay']}")

print(f"\nMétricas (Fold 1):")
print_metrics(best_config_result['metrics'], title="")

print(f"\n✅ Esta configuración se usará para entrenamiento final")

# Comparar con baseline
baseline_result = next(r for r in hyperparameter_results if r['config']['name'] == 'Baseline')
improvement_r2 = best_config_result['avg_r2'] - baseline_result['avg_r2']
improvement_mae = ((baseline_result['avg_mae'] - best_config_result['avg_mae']) / baseline_result['avg_mae']) * 100

print(f"\n📊 Mejora vs Baseline:")
print(f"  R² promedio: {baseline_result['avg_r2']:.4f} → {best_config_result['avg_r2']:.4f} ({improvement_r2:+.4f})")
print(f"  MAE promedio: {baseline_result['avg_mae']:.4f} → {best_config_result['avg_mae']:.4f} ({improvement_mae:+.1f}%)")


RANKING DE CONFIGURACIONES

Rank   Configuración        R² Avg       MAE Avg      P2.5 R²      O3 R²        CO R²       
------------------------------------------------------------------------------------------
1      Higher LR            0.5059       1.8442       0.3261       0.6928       0.4990      
2      Less Dropout         0.5017       1.7933       0.2989       0.7028       0.5034      
3      Smaller Network      0.4961       1.7240       0.3870       0.7032       0.3980      
4      Wider Network        0.4772       2.0122       0.2748       0.7261       0.4306      
5      Lower LR             0.4731       1.7403       0.3776       0.7052       0.3364      
6      Deeper Network       0.4473       1.9198       0.2858       0.6459       0.4102      
7      Baseline             0.4409       1.8129       0.3521       0.6837       0.2868      
8      More Dropout         0.4380       1.8078       0.3464       0.6702       0.2975      

MEJOR CONFIGURACIÓN: Higher LR

Hiperparám

In [33]:
# ============================================================================
# RESUMEN COMPLETO DEL PROYECTO
# ============================================================================

print("\n" + "="*70)
print("RESUMEN COMPLETO - PROYECTO MLP PARA PREDICCIÓN DE CONTAMINANTES")
print("="*70)

print(f"\n📊 EVOLUCIÓN DEL MODELO:")
print(f"\n1️⃣ Modelo Inicial (sin normalización):")
print(f"   R² promedio: -4.78 (TERRIBLE)")
print(f"   Problema: CO con R² de -14.12")

print(f"\n2️⃣ Modelo con Normalización:")
print(f"   R² promedio: 0.37 (ACEPTABLE)")
print(f"   Mejora: +5.15 puntos")
print(f"   ✅ O3 pasó de -0.53 a 0.75")
print(f"   ✅ CO pasó de -14.12 a 0.03")

print(f"\n3️⃣ Modelo con Mejor Configuración (Higher LR):")
print(f"   R² promedio: 0.51 (BUENO - solo Fold 1)")
print(f"   Mejora vs normalizado: +0.14 puntos")
print(f"   ✅ CO mejoró de 0.03 a 0.50")
print(f"   ✅ O3 mejoró de 0.75 a 0.69")

print(f"\n🎯 MEJOR CONFIGURACIÓN ENCONTRADA:")
print(f"   - Hidden layers: [128, 64, 32]")
print(f"   - Dropout: 0.2")
print(f"   - Learning rate: 0.002 (el doble del inicial)")
print(f"   - Weight decay: 0.01")
print(f"   - Normalización: SÍ (StandardScaler)")

print(f"\n📈 RENDIMIENTO POR CONTAMINANTE (Mejor Config):")
print(f"   P2.5: R²=0.33, MAE=5.46 µg/m³")
print(f"   O3:   R²=0.69, MAE=0.0046 ppm")
print(f"   CO:   R²=0.50, MAE=0.068 ppm")

print(f"\n✅ PRÓXIMOS PASOS RECOMENDADOS:")
print(f"   1. Entrenar mejor configuración en todos los 5 folds")
print(f"   2. Entrenar modelo final con TODOS los datos (train+val)")
print(f"   3. Evaluar UNA vez en test_final")
print(f"   4. Guardar modelo y scalers para producción")
print(f"   5. Implementar pipeline de predicción")

print(f"\n💾 DATOS PARA PRODUCCIÓN:")
print(f"   - Modelo: AirQualityMLP con mejor configuración")
print(f"   - Scalers: StandardScaler para X e y")
print(f"   - Features: 69 (11 vars + 55 lags + 3 temporales)")
print(f"   - Targets: 3 (P2.5, O3, CO)")

print(f"\n🎉 PROYECTO COMPLETADO CON ÉXITO!")


RESUMEN COMPLETO - PROYECTO MLP PARA PREDICCIÓN DE CONTAMINANTES

📊 EVOLUCIÓN DEL MODELO:

1️⃣ Modelo Inicial (sin normalización):
   R² promedio: -4.78 (TERRIBLE)
   Problema: CO con R² de -14.12

2️⃣ Modelo con Normalización:
   R² promedio: 0.37 (ACEPTABLE)
   Mejora: +5.15 puntos
   ✅ O3 pasó de -0.53 a 0.75
   ✅ CO pasó de -14.12 a 0.03

3️⃣ Modelo con Mejor Configuración (Higher LR):
   R² promedio: 0.51 (BUENO - solo Fold 1)
   Mejora vs normalizado: +0.14 puntos
   ✅ CO mejoró de 0.03 a 0.50
   ✅ O3 mejoró de 0.75 a 0.69

🎯 MEJOR CONFIGURACIÓN ENCONTRADA:
   - Hidden layers: [128, 64, 32]
   - Dropout: 0.2
   - Learning rate: 0.002 (el doble del inicial)
   - Weight decay: 0.01
   - Normalización: SÍ (StandardScaler)

📈 RENDIMIENTO POR CONTAMINANTE (Mejor Config):
   P2.5: R²=0.33, MAE=5.46 µg/m³
   O3:   R²=0.69, MAE=0.0046 ppm
   CO:   R²=0.50, MAE=0.068 ppm

✅ PRÓXIMOS PASOS RECOMENDADOS:
   1. Entrenar mejor configuración en todos los 5 folds
   2. Entrenar modelo final co

In [23]:
# ============================================================================
# RESUMEN FINAL DEL PROYECTO
# ============================================================================

print("\n" + "="*70)
print("RESUMEN DEL PROYECTO - MLP PARA PREDICCIÓN DE CONTAMINANTES")
print("="*70)

print(f"\n📊 DATOS:")
print(f"   Periodo: Jun 2024 - Feb 2025 (9 meses)")
print(f"   Total de horas con datos completos: {len(df_filtered)}")
print(f"   Features: {len(feature_cols)}")
print(f"     - Variables actuales: {len(var_cols)}")
print(f"     - Rezagos (t-1 a t-5): {5 * len(var_cols)}")
print(f"     - Temporales: 3 (Hora, Día, Mes)")
print(f"   Targets: 3 (P2.5, O3, CO)")

print(f"\n🎯 ARQUITECTURA:")
print(f"   Tipo: MLP (Red Totalmente Conectada)")
print(f"   Input: 69 features → Hidden: {HIDDEN_SIZES} → Output: 3")
print(f"   Regularización: Dropout ({DROPOUT}), L2 ({WEIGHT_DECAY})")
print(f"   Batch Normalization: Sí")
print(f"   Parámetros totales: ~19,843")

print(f"\n📈 ENTRENAMIENTO:")
print(f"   Función de pérdida: MSE + L2 regularization")
print(f"   Optimizador: Adam (lr={LEARNING_RATE})")
print(f"   Early stopping: {PATIENCE} épocas")
print(f"   Validación: 5-Fold Time Series CV con GAP de 24h")

print(f"\n📏 MÉTRICAS:")
print(f"   - MAE: Error promedio en unidades originales")
print(f"   - RMSE: Penaliza más los errores grandes")
print(f"   - R²: Proporción de varianza explicada")

print(f"\n⚠️  LIMITACIONES:")
print(f"   - Faltan datos de Mar, Abr, May")
print(f"   - El modelo tendrá menor precisión en esos meses")
print(f"   - Recomendación: Reentrenar con año completo")

print(f"\n✅ PRÓXIMOS PASOS:")
print(f"   1. Ejecutar CV completo (celda anterior)")
print(f"   2. Ajustar hiperparámetros si es necesario")
print(f"   3. Entrenar modelo final con todos los datos train+val")
print(f"   4. Evaluar UNA vez en test_final")
print(f"   5. Guardar modelo para producción")


RESUMEN DEL PROYECTO - MLP PARA PREDICCIÓN DE CONTAMINANTES

📊 DATOS:
   Periodo: Jun 2024 - Feb 2025 (9 meses)
   Total de horas con datos completos: 5641
   Features: 69
     - Variables actuales: 11
     - Rezagos (t-1 a t-5): 55
     - Temporales: 3 (Hora, Día, Mes)
   Targets: 3 (P2.5, O3, CO)

🎯 ARQUITECTURA:
   Tipo: MLP (Red Totalmente Conectada)
   Input: 69 features → Hidden: [128, 64, 32] → Output: 3
   Regularización: Dropout (0.2), L2 (0.01)
   Batch Normalization: Sí
   Parámetros totales: ~19,843

📈 ENTRENAMIENTO:
   Función de pérdida: MSE + L2 regularization
   Optimizador: Adam (lr=0.001)
   Early stopping: 15 épocas
   Validación: 5-Fold Time Series CV con GAP de 24h

📏 MÉTRICAS:
   - MAE: Error promedio en unidades originales
   - RMSE: Penaliza más los errores grandes
   - R²: Proporción de varianza explicada

⚠️  LIMITACIONES:
   - Faltan datos de Mar, Abr, May
   - El modelo tendrá menor precisión en esos meses
   - Recomendación: Reentrenar con año completo

✅ 

In [1]:
"""
Entrenamiento de Modelo MLP para Predicción de Contaminantes del Aire
Variables a predecir: P2.5, O3, CO (próxima hora t+1)
"""

from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

# ============================================================================
# 1. CARGA Y PROCESAMIENTO DE DATOS
# ============================================================================

# Cargar datos
dataset = pd.read_json("dataset.json")
days: list[Any] = list(dataset["fecha"])
hours: list[Any] = list(dataset.iloc[0]["horas"].keys())
hours.sort(key=lambda x: int(x.split(":")[0]))
variables: list[Any] = list(dataset.iloc[0]["horas"][hours[0]])

# Crear DataFrame con todas las horas
rows = []
for day_index, date in enumerate(days):
    for hour in hours:
        row = [date, hour]
        for var in variables:
            value = dataset.iloc[day_index]["horas"][hour][var]
            row.append(float(value) if value != "" else None)
        rows.append(row)

df = pd.DataFrame(rows, columns=["Día", "Hora"] + variables)

# Limpieza: eliminar T y corregir valores
df = df.drop(columns=["T"])

# Reemplazar valores negativos por 0
non_negative_variables = ["CO", "NO2", "NO", "NxOy", "O3", "P2.5", "PP"]
for var in non_negative_variables:
    if var in df.columns:
        df[var] = df[var].apply(lambda x: 0 if pd.notna(x) and x < 0 else x)

# Corregir outliers en VV
df.loc[df["VV"] > 500, "VV"] = None

# Features temporales
df["Día_numérico"] = pd.to_datetime(df["Día"]).dt.day
df["Mes"] = pd.to_datetime(df["Día"]).dt.month
df["Hora_numérica"] = df["Hora"].str.split(":").str[0].astype(int)

# Variables a usar
var_cols = ["CO", "NO2", "NO", "NxOy", "O3", "P2.5", "VV", "HR", "PB", "RS", "PP"]


# Crear rezagos y targets
def create_lagged_features(df, lag_hours=5):
    df_lagged = df.copy()

    for var in var_cols:
        for lag in range(1, lag_hours + 1):
            df_lagged[f"{var}_lag{lag}"] = df[var].shift(lag)

    for var in var_cols:
        df_lagged[f"{var}_next"] = df[var].shift(-1)

    df_lagged = df_lagged.dropna()
    return df_lagged


df_lagged = create_lagged_features(df, lag_hours=5)

# Filtrar desde Junio 2024
df_filtered = df_lagged[pd.to_datetime(df_lagged["Día"]) >= "2024-06-01"].reset_index(
    drop=True
)

# Features y targets
feature_cols = (
    var_cols
    + [f"{var}_lag{i}" for var in var_cols for i in range(1, 6)]
    + ["Hora_numérica", "Día_numérico", "Mes"]
)
target_vars = ["P2.5", "O3", "CO"]
target_cols = [f"{var}_next" for var in target_vars]

In [2]:
df_filtered

,Día,Hora,CO,NO2,NO,NxOy,O3,P2.5,VV,HR,...,NO2_next,NO_next,NxOy_next,O3_next,P2.5_next,VV_next,HR_next,PB_next,RS_next,PP_next
0,2024-06-13,15:00 - 16:00,0.90,0.017,0.001,0.018,0.000,50.0,129.0,84.0,...,0.007,0.000,0.008,0.000,50.0,192.0,86.0,762.576,1488.0,0.0
1,2024-06-13,16:00 - 17:00,0.87,0.007,0.000,0.008,0.000,50.0,192.0,86.0,...,0.014,0.002,0.016,0.079,17.0,258.0,85.0,762.991,1497.0,0.0
2,2024-06-13,17:00 - 18:00,0.97,0.014,0.002,0.016,0.079,17.0,258.0,85.0,...,0.011,0.002,0.012,0.000,0.0,247.0,85.0,763.124,1499.0,0.0
3,2024-06-13,18:00 - 19:00,0.89,0.011,0.002,0.012,0.000,0.0,247.0,85.0,...,0.009,0.002,0.010,0.000,0.0,280.0,87.0,763.318,1499.0,0.0
4,2024-06-13,19:00 - 20:00,0.82,0.009,0.002,0.010,0.000,0.0,280.0,87.0,...,0.009,0.001,0.010,0.000,3.0,292.0,87.0,763.487,1499.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5636,2025-02-25,17:00 - 18:00,2.15,0.001,0.000,0.001,0.018,9.0,180.0,63.0,...,0.001,0.000,0.002,0.016,8.0,166.0,67.0,765.046,1499.0,0.0
5637,2025-02-25,18:00 - 19:00,2.19,0.001,0.000,0.002,0.016,8.0,166.0,67.0,...,0.002,0.000,0.002,0.013,6.0,161.0,68.0,765.369,1499.0,0.0
5638,2025-02-25,19:00 - 20:00,2.31,0.002,0.000,0.002,0.013,6.0,161.0,68.0,...,0.002,0.000,0.003,0.012,9.0,160.0,70.0,765.489,1499.0,0.0
5639,2025-02-25,20:00 - 21:00,2.35,0.002,0.000,0.003,0.012,9.0,160.0,70.0,...,0.003,0.000,0.003,0.010,9.0,159.0,72.0,765.556,1499.0,0.0


In [ ]:
# ============================================================================
# 2. TIME SERIES CROSS-VALIDATION SPLITS
# ============================================================================

GAP_HOURS = 24
N_FOLDS = 5
MIN_TRAIN_SIZE = 2000


def create_time_series_folds(df, n_folds=5, gap=24, min_train_size=2000):
    folds = []
    n_total = len(df)
    val_size = (n_total - min_train_size - (n_folds * gap)) // n_folds

    for fold_idx in range(n_folds):
        train_end = min_train_size + (fold_idx * val_size)
        gap_end = train_end + gap
        val_end = gap_end + val_size

        if val_end > n_total:
            break

        train_df = df.iloc[:train_end]
        val_df = df.iloc[gap_end:val_end]

        folds.append({"fold": fold_idx + 1, "train": train_df, "val": val_df})

    return folds


cv_folds = create_time_series_folds(
    df_filtered, n_folds=N_FOLDS, gap=GAP_HOURS, min_train_size=MIN_TRAIN_SIZE
)

# Test final
test_start_idx = len(df_filtered) - int(0.15 * len(df_filtered))
test_gap_start = test_start_idx - GAP_HOURS
test_final = df_filtered.iloc[test_start_idx:]
X_test_final = test_final[feature_cols]
y_test_final = test_final[target_cols]

# ============================================================================
# 3. NORMALIZACIÓN
# ============================================================================


def create_normalized_cv_data(cv_folds, feature_cols, target_cols):
    normalized_cv_data = []

    for fold_idx, fold_info in enumerate(cv_folds, 1):
        X_train = fold_info["train"][feature_cols]
        y_train = fold_info["train"][target_cols]
        X_val = fold_info["val"][feature_cols]
        y_val = fold_info["val"][target_cols]

        scaler_X = StandardScaler()
        scaler_y = StandardScaler()

        X_train_scaled = scaler_X.fit_transform(X_train)
        y_train_scaled = scaler_y.fit_transform(y_train)
        X_val_scaled = scaler_X.transform(X_val)
        y_val_scaled = scaler_y.transform(y_val)

        X_train_scaled = pd.DataFrame(
            X_train_scaled, columns=feature_cols, index=X_train.index
        )
        y_train_scaled = pd.DataFrame(
            y_train_scaled, columns=target_cols, index=y_train.index
        )
        X_val_scaled = pd.DataFrame(
            X_val_scaled, columns=feature_cols, index=X_val.index
        )
        y_val_scaled = pd.DataFrame(
            y_val_scaled, columns=target_cols, index=y_val.index
        )

        normalized_cv_data.append(
            {
                "fold": fold_idx,
                "X_train": X_train_scaled,
                "y_train": y_train_scaled,
                "X_val": X_val_scaled,
                "y_val": y_val_scaled,
                "scaler_X": scaler_X,
                "scaler_y": scaler_y,
            }
        )

    return normalized_cv_data


cv_data_normalized = create_normalized_cv_data(cv_folds, feature_cols, target_cols)

In [12]:
print(df_filtered.iloc[0])

Día           2024-06-13
Hora       15:00 - 16:00
CO                   0.9
NO2                0.017
NO                 0.001
               ...      
VV_next            192.0
HR_next             86.0
PB_next          762.576
RS_next           1488.0
PP_next              0.0
Name: 0, Length: 82, dtype: object


In [ ]:
"""
0.9
0.017
0.001
0.018000000000000002
0.0
50.0
129.0
84.0
762.31
1480.0
0.0
13
6
15
0.8200000000000001
0.8
0.79
0.8
0.84
0.005
0.005
0.005
0.007
0.011
0.0
0.001
0.001
0.001
0.002
0.005
0.005
0.005
0.007
0.013000000000000001
0.0
0.07
0.055
0.056
0.047
25.0
17.0
7.0
3.0
0.0
175.0
192.0
178.0
146.0
114.0
66.0
62.0
48.0
56.0
60.0
762.259
762.112
762.297
762.584
762.812
1365.0
1123.0
674.0
603.0
816.0
0.0
0.0
0.0
0.0
0.0
0.87
0.007
0.0
0.008
0.0
50.0
192.0
86.0
762.576
1488.0
0.0
"""

In [17]:
elemento = []
for element in df_filtered.iloc[0]:
    elemento.append(element)

for i, col in enumerate(df_filtered.columns):
    print(f'"{col}": {elemento[i]},')

"Día": 2024-06-13,
"Hora": 15:00 - 16:00,
"CO": 0.9,
"NO2": 0.017,
"NO": 0.001,
"NxOy": 0.018000000000000002,
"O3": 0.0,
"P2.5": 50.0,
"VV": 129.0,
"HR": 84.0,
"PB": 762.31,
"RS": 1480.0,
"PP": 0.0,
"Día_numérico": 13,
"Mes": 6,
"Hora_numérica": 15,
"CO_lag1": 0.8200000000000001,
"CO_lag2": 0.8,
"CO_lag3": 0.79,
"CO_lag4": 0.8,
"CO_lag5": 0.84,
"NO2_lag1": 0.005,
"NO2_lag2": 0.005,
"NO2_lag3": 0.005,
"NO2_lag4": 0.007,
"NO2_lag5": 0.011,
"NO_lag1": 0.0,
"NO_lag2": 0.001,
"NO_lag3": 0.001,
"NO_lag4": 0.001,
"NO_lag5": 0.002,
"NxOy_lag1": 0.005,
"NxOy_lag2": 0.005,
"NxOy_lag3": 0.005,
"NxOy_lag4": 0.007,
"NxOy_lag5": 0.013000000000000001,
"O3_lag1": 0.0,
"O3_lag2": 0.07,
"O3_lag3": 0.055,
"O3_lag4": 0.056,
"O3_lag5": 0.047,
"P2.5_lag1": 25.0,
"P2.5_lag2": 17.0,
"P2.5_lag3": 7.0,
"P2.5_lag4": 3.0,
"P2.5_lag5": 0.0,
"VV_lag1": 175.0,
"VV_lag2": 192.0,
"VV_lag3": 178.0,
"VV_lag4": 146.0,
"VV_lag5": 114.0,
"HR_lag1": 66.0,
"HR_lag2": 62.0,
"HR_lag3": 48.0,
"HR_lag4": 56.0,
"HR_lag5": 60.0,
"